In [ ]:
import numpy as np

np.set_printoptions(suppress=True)

#generate randomly distributed params
params=np.random.uniform(low=-50, high=150,size=10000)

#outlier
#outlier is a value that is abnormal wrt other values
#in this case other values liew btw -50 and 150 but the outlier is 1000
params[-1]=1000

params=np.round(params,2)

print(params)

[  58.09   -4.86  -44.91 ...  138.09   92.15 1000.  ]


In [5]:
#This is the same code as before
# Min max strategy where alpha=max and beta=min
def clamp(params_q: np.array, lower_bound: int, upper_bound: int)->np.array:
    params_q[params_q<lower_bound]=lower_bound
    params_q[params_q>upper_bound]=upper_bound
    return params_q

def asymmetric_quantization(params: np.array, bits: int) -> tuple[np.array,float,int]:
    alpha=np.max(params)
    beta=np.min(params)
    scale=(alpha-beta)/(2**bits-1)
    zero=-1*np.round(beta/scale)
    lower_bound=0
    upper_bound=2**bits-1
    
    quantized=clamp(np.round(params/scale + zero),lower_bound,upper_bound).astype(np.int32)
    return quantized, scale, zero

def asymmetric_dequantization(params_q: np.array, scale:float, zero:int)-> np.array:
    return(params_q-zero)*scale


In [4]:
#Percentile strategy

def asymmetric_quantization_percentile(params: np.array, bits: int,percentile: float= 99.99) -> tuple[np.array,float,int]:
#find the percentile value
    alpha=np.percentile(params,percentile)
    beta=np.percentile(params,100-percentile)
    scale=(alpha-beta)/(2**bits-1)
    zero=-1*np.round(beta/scale)
    lower_bound=0
    upper_bound=2**bits-1
    #quantize the parameters
    quantized=clamp(np.round(params/scale + zero),lower_bound,upper_bound).astype(np.int32)
    return quantized, scale, zero

def asymmetric_dequantization(params_q: np.array, scale:float, zero:int)-> np.array:
    return(params_q-zero)*scale


In [8]:
(asymmetric_q, asymmetric_scale,asymmetric_zero)=asymmetric_quantization(params,8)
(asymmetric_qp, asymmetric_scalep,asymmetric_zerop)=asymmetric_quantization_percentile(params,8)

print(f"Original: ")
print(np.round(params,2))

print(f"Asymmetric (min, max) Scale: {asymmetric_scale}, Zero: {asymmetric_zero}")
print(asymmetric_q)

print(f"Asymmetric (percentile) Scale: {asymmetric_scalep}, Zero: {asymmetric_zerop}")
print(asymmetric_qp)

Original: 
[  58.09   -4.86  -44.91 ...  138.09   92.15 1000.  ]
Asymmetric (min, max) Scale: 4.117607843137255, Zero: 12.0
[ 26  11   1 ...  46  34 255]
Asymmetric (percentile) Scale: 0.7843333607819561, Zero: 64.0
[138  58   7 ... 240 181 255]


In [ ]:
#dequantize
params_deq_asymmetric=asymmetric_dequantization(asymmetric_q,asymmetric_scale, asymmetric_zero)
#we use the same formula as min max because the only difference was during computation of alpha and beta
params_deq_asymmetric_p=asymmetric_dequantization(asymmetric_qp,asymmetric_scalep, asymmetric_zerop)

print(f"Original: ")
print(np.round(params, 2))
print(" ")

print(f"Dequantized (Min Max): ")
print(np.round(params_deq_asymmetric,2))
print(" ")

print(f"Dequantized (Percentile): ")
print(np.round(params_deq_asymmetric_p,2))
print(" ")
 

Original: 
[  58.09   -4.86  -44.91 ...  138.09   92.15 1000.  ]
 
Dequantized (Min Max): 
[  57.65   -4.12  -45.29 ...  140.     90.59 1000.58]
 
Dequantized (Percentile): 
[ 58.04  -4.71 -44.71 ... 138.04  91.77 149.81]
 


In [15]:
def quantization_error(params: np.array, params_q: np.array):
    return np.mean((params-params_q)**2)
print(f"{"error(min max): "}{np.round(quantization_error(params[:-1], params_deq_asymmetric[:-1]),2)}")
print(f"{"error(percentile): "}{np.round(quantization_error(params[:-1], params_deq_asymmetric_p[:-1]),2)}")


error(min max): 1.41
error(percentile): 0.05
